# Run a project script on a free Colab GPU

The laptop this project was built on has no CUDA GPU, so the slow steps (captioning, answering, the full pipeline) were run here on a T4 with the same scripts.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save.

On your own machine, zip the scripts and the data they need from the project root, for example:

```
tar -a -c -f upload.zip *.py data/pages data/pages.jsonl data/captions.jsonl data/questions.jsonl data/index
```

## 1. Choose what to run

In [ ]:
# One of:
#   'caption.py'            -> data/captions.jsonl
#   'answer.py run'         -> data/answers.jsonl
#   'rag.py eval basic'     -> data/rag_answers_basic.jsonl
#   'rag.py eval evidence'  -> data/rag_answers_evidence.jsonl
COMMAND = 'rag.py eval basic'
RESULT = 'data/rag_answers_basic.jsonl'

## 2. Check the GPU, install Ollama and start it

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import os, subprocess, time
log = open('ollama.log', 'w')
# A page image plus its text needs more room than Ollama's default 4096-token window.
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH='8192')
subprocess.Popen(['ollama', 'serve'], stdout=log, stderr=log, env=env)
time.sleep(8)
!ollama --version

## 3. Download the vision and embedding models (a few minutes)

In [ ]:
!ollama pull qwen2.5vl:7b
!ollama pull nomic-embed-text

## 4. Upload `upload.zip`

In [ ]:
from google.colab import files
import zipfile, pathlib

uploaded = files.upload()
name = next(iter(uploaded))

with zipfile.ZipFile(name) as z:
    for info in z.infolist():
        target = pathlib.Path(info.filename.replace('\\', '/'))  # tolerate Windows-style paths
        if info.is_dir() or not target.name:
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(z.read(info))

assert pathlib.Path('llm.py').exists() and pathlib.Path('data/pages.jsonl').exists(), 'check the zip contents'
print('PNG pages found:', len(list(pathlib.Path('data/pages').rglob('*.png'))))

## 5. Run

Every script is resumable: it skips work already recorded in its result file.

In [ ]:
!pip -q install openai numpy
!python -u {COMMAND}
!ollama ps

## 6. Download the result

Put the downloaded file into the project's `data` folder.

In [ ]:
files.download(RESULT)